In [1]:
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from dotenv import load_dotenv

load_dotenv()

llm = init_chat_model("openai:gpt-4o")

In [2]:
class State(MessagesState):
    custom_stuff: str

graph_builder = StateGraph(State)

In [3]:
@tool
def get_weather(city: str):
    """ Gets weather in city """
    return f"The weather in {city} is sunny."

llm_with_tools = llm.bind_tools(tools=[get_weather]) # LLM에게 Tool을 설명해줌.

def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return { "messages": [response] }
    

In [4]:
tool_node = ToolNode(
    tools=[
        get_weather,
    ],
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile()

In [ ]:
result = graph.invoke({
    "messages": [
        {"role": "user", "content": "What is the weather in Seoul?"}
    ]
})

print(result)
graph